# Assignment — Bloc 7
### Mini-repte: entrenar i avaluar classificadors

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Colab el carregarà directament des de GitHub. Per guardar els teus canvis: `File → Save a copy in Drive`.

Completa les funcions marcades amb `# TODO`. Cada una té el seu autotest: si l'autotest passa (✅), aquella part ja és correcta.

In [ ]:
import os
if not os.path.exists('/content/tp2'):
    os.system('git clone -q https://github.com/jcomajuncosas/tp2.git /content/tp2')

DATASET_DIR = "/content/tp2/06_analisi_ml/sessio11_fft_features/dataset"
CSV_PATH = "features_percussio.csv"

if not os.path.exists(CSV_PATH):
    import librosa, glob, warnings
    warnings.filterwarnings('ignore')
    rows = []
    for categoria in ['kick', 'hihat']:
        for f in sorted(glob.glob(f"{DATASET_DIR}/{categoria}/*.wav")):
            audio, sr = librosa.load(f, sr=None)
            n_fft = min(2048, len(audio))
            centroid = librosa.feature.spectral_centroid(y=audio, sr=sr, n_fft=n_fft).mean()
            zcr = librosa.feature.zero_crossing_rate(audio).mean()
            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13, n_fft=n_fft).mean(axis=1)
            row = {'fitxer': f.split('/')[-1], 'classe': categoria, 'centroid': centroid, 'zcr': zcr}
            for i, val in enumerate(mfcc):
                row[f'mfcc_{i}'] = val
            rows.append(row)
    import pandas as pd
    pd.DataFrame(rows).to_csv(CSV_PATH, index=False)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

df = pd.read_csv(CSV_PATH)
print(f"Dataset carregat: {df.shape[0]} sons")

## TODO 1 — `prepara_dades(df, features, test_size=0.3)`

Completa una funció que, donat el DataFrame i una llista de noms de columnes (features), preparí `X`/`y` i en faci el train/test split.

In [ ]:
def prepara_dades(df, features, test_size=0.3, random_state=42):
    """
    A partir del DataFrame 'df' i una llista 'features' (noms de columnes,
    p.ex. ['centroid', 'zcr']), prepara les dades per entrenar:

      1. X = els valors de les columnes 'features' (df[features].values)
      2. y = 0 si la classe és 'kick', 1 si és 'hihat'
         (pots fer-ho amb (df['classe'] == 'hihat').astype(int).values)
      3. fes el train/test split amb train_test_split(), usant
         test_size, random_state, i stratify=y (important amb pocs sons)

    Retorna: X_train, X_test, y_train, y_test (en aquest ordre,
    és l'ordre que retorna train_test_split)
    """
    # TODO: implementa els 3 passos de la docstring
    X = None  # <-- substitueix
    y = None  # <-- substitueix
    return None  # <-- substitueix: retorna el resultat de train_test_split(...)

In [ ]:
def _test_prepara_dades():
    print("Test 1: prepara_dades()...")

    resultat = prepara_dades(df, ['centroid', 'zcr'])
    assert resultat is not None, "❌ prepara_dades() no retorna res"
    assert len(resultat) == 4, "❌ Hauria de retornar 4 valors (X_train, X_test, y_train, y_test)"

    X_train, X_test, y_train, y_test = resultat
    assert X_train.shape[1] == 2, f"❌ X_train hauria de tenir 2 columnes (centroid, zcr), en té {X_train.shape[1]}"
    assert len(X_train) + len(X_test) == len(df), "❌ El total de train+test hauria de ser igual al dataset complet"
    assert set(np.unique(y_train)) <= {0, 1}, "❌ y hauria de contenir només 0 (kick) i 1 (hihat)"

    # Comprovem stratify: la proporció de classes a train hauria de ser similar a la del dataset
    prop_dataset = (df['classe'] == 'hihat').mean()
    prop_train = y_train.mean()
    assert abs(prop_dataset - prop_train) < 0.15, (
        "❌ La proporció de classes a train s'allunya massa de la del dataset original "
        "— revisa que facis servir stratify=y"
    )

    print(f"   Train: {len(X_train)} mostres, Test: {len(X_test)} mostres")
    print("✅ prepara_dades() correcte\n")

_test_prepara_dades()

## TODO 2 — `entrena_i_avalua(clf, X_train, X_test, y_train, y_test)`

Completa una funció que entreni un classificador donat i en retorni l'accuracy sobre el conjunt de test.

In [ ]:
def entrena_i_avalua(clf, X_train, X_test, y_train, y_test):
    """
    Entrena 'clf' amb (X_train, y_train), prediu sobre X_test, i retorna
    l'accuracy (accuracy_score) comparant la predicció amb y_test.

    Patró (igual que a patches_bloc7.ipynb):
      1. clf.fit(X_train, y_train)
      2. y_pred = clf.predict(X_test)
      3. return accuracy_score(y_test, y_pred)
    """
    # TODO: implementa els 3 passos de la docstring
    pass  # <-- substitueix

In [ ]:
def _test_entrena_i_avalua():
    print("Test 2: entrena_i_avalua()...")

    X_train, X_test, y_train, y_test = prepara_dades(df, ['centroid', 'zcr'])
    clf = KNeighborsClassifier(n_neighbors=3)

    acc = entrena_i_avalua(clf, X_train, X_test, y_train, y_test)
    assert acc is not None, "❌ entrena_i_avalua() no retorna res"
    assert isinstance(acc, (float, np.floating)), f"❌ Hauria de retornar un float, ha retornat {type(acc)}"
    assert 0.0 <= acc <= 1.0, f"❌ L'accuracy ha d'estar entre 0 i 1, ha donat {acc}"

    # Amb 'centroid'/'zcr' (separació molt clara) esperem una accuracy alta
    assert acc >= 0.7, f"❌ Amb aquestes features esperàvem una accuracy alta, ha donat {acc:.0%}"

    print(f"   Accuracy KNN: {acc:.0%}")
    print("✅ entrena_i_avalua() correcte\n")

_test_entrena_i_avalua()

## Comparant els tres classificadors

Amb les dues funcions ja fetes, comparem KNN, Decision Tree i SVM (no cal tocar res d'aquí en avall).

In [ ]:
X_train, X_test, y_train, y_test = prepara_dades(df, ['centroid', 'zcr'])

classificadors = {
    'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'SVM (rbf)': SVC(kernel='rbf', random_state=42),
}

for nom, clf in classificadors.items():
    acc = entrena_i_avalua(clf, X_train, X_test, y_train, y_test)
    print(f"{nom}: {acc:.0%}")

print()
print("Recordatori: amb només 29 mostres, aquests números mostren que el")
print("PROCEDIMENT funciona correctament — no són una comparativa estadísticament")
print("robusta de quin algorisme és 'millor' en general.")

In [ ]:
def _test_comparativa_final():
    print("Test 3: comparativa amb els 3 classificadors...")
    resultats = {}
    for nom, clf in classificadors.items():
        acc = entrena_i_avalua(clf, X_train, X_test, y_train, y_test)
        resultats[nom] = acc

    assert len(resultats) == 3, "❌ Haurien d'haver-hi resultats per als 3 classificadors"
    assert all(0.0 <= v <= 1.0 for v in resultats.values()), "❌ Totes les accuracies han d'estar entre 0 i 1"

    print("✅ Comparativa completa i correcta\n")

_test_comparativa_final()

## Per provar tu mateix (opcional)

Prova `prepara_dades()` amb un altre parell de features (per exemple `['mfcc_1', 'mfcc_2']`, com a `patches_bloc7.ipynb`) i compara les accuracies. Canvien molt respecte a `centroid`/`zcr`?

## Recordatori

Aquest és l'últim contingut purament tècnic abans del projecte final. El procediment d'avui (preparar dades, entrenar, avaluar, interpretar amb cura) és transferible a qualsevol problema de classificació, no només a sons de percussió.